<a href="https://colab.research.google.com/github/swethaukkarde/Neural-networks-and-deep-learning/blob/main/Exp_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Problem Statement:
# A hospital wants an AI model that identifies pneumonia from chest X-rays
# but has limited training data. Use transfer learning with VGG16/ResNet
# and compare performance with training from scratch.

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

# -------------------------------
# Step 1: Load Dataset (Reduced)
# -------------------------------
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Use small subset to avoid RAM crash
x_train = x_train[:5000]
y_train = y_train[:5000]

# Explicitly convert numpy arrays to TensorFlow Tensors and cast to float32
x_train = tf.cast(x_train, tf.float32)
x_test = tf.cast(x_test, tf.float32)

# Add channel dimension and convert to 3 channels
x_train = tf.image.grayscale_to_rgb(tf.expand_dims(x_train, axis=-1))
x_test = tf.image.grayscale_to_rgb(tf.expand_dims(x_test, axis=-1))

# Resize (smaller than 224)
x_train = tf.image.resize(x_train, (96,96)) / 255.0
x_test = tf.image.resize(x_test, (96,96)) / 255.0

# -------------------------------
# Step 2: Transfer Learning (Light Model)
# -------------------------------
base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(96,96,3))
base.trainable = False

model_tl = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(10, activation='softmax')
])

model_tl.compile(optimizer='adam',
                 loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])

model_tl.fit(x_train, y_train, epochs=2, verbose=1)

# -------------------------------
# Step 3: CNN from Scratch
# -------------------------------
model_s = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(96,96,3)),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model_s.compile(optimizer='adam',
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])

model_s.fit(x_train, y_train, epochs=2, verbose=1)

# -------------------------------
# Step 4: Compare Accuracy
# -------------------------------
acc_tl = model_tl.evaluate(x_test, y_test, verbose=0)[1]
acc_s = model_s.evaluate(x_test, y_test, verbose=0)[1]

print("Transfer Learning Accuracy:", acc_tl)
print("Scratch Model Accuracy:", acc_s)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/2
157/157 ━━━━━━━━━━━━━━━━━━━━ 36s 197ms/step - accuracy: 0.8134 - loss: 0.6424
Epoch 2/2
157/157 ━━━━━━━━━━━━━━━━━━━━ 40s 190ms/step - accuracy: 0.9412 - loss: 0.2167
Epoch 1/2


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


157/157 ━━━━━━━━━━━━━━━━━━━━ 37s 227ms/step - accuracy: 0.8508 - loss: 0.4883
Epoch 2/2
157/157 ━━━━━━━━━━━━━━━━━━━━ 35s 221ms/step - accuracy: 0.9430 - loss: 0.1927
Transfer Learning Accuracy: 0.9383000135421753
Scratch Model Accuracy: 0.9296000003814697
